# 9장 - 화자 구분 + 타임스탬프 회의록

원본 파일: `chap09/stt_with_speakers.py`

In [2]:
"""화자 구분 + 타임스탬프까지 한 번에 (회의록 만들기)

OpenRouter의 오디오 채팅 모델(gemini-2.5-flash)에 대화 음성을 넣으면,
"누가(화자) / 언제(시작~종료 시각) / 뭐라고(텍스트)"를 한 번의 호출로 정리해준다.
기존에는 로컬 Whisper(STT) + pyannote(화자 분리) 두 모델을 따로 돌려야 했지만,
이제 API 호출 한 번으로 끝난다.
"""
from openai import OpenAI
from dotenv import load_dotenv
import os
import base64
import json
import re
import pandas as pd

load_dotenv()

BASE_DIR = (os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd())

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
)

AUDIO_MODEL = "google/gemini-2.5-flash"

INSTRUCTION = """이 오디오는 여러 사람의 대화입니다.
각 발언을 화자별로 구분하고, 시작/종료 시각을 초 단위 숫자(소수점 허용)로 표기해서 JSON 배열로만 출력하세요.
형식: [{"start": 1.7, "end": 49.8, "speaker": "화자A", "text": "..."}]
JSON 외의 다른 텍스트나 코드블록 표시는 절대 쓰지 마세요."""

In [3]:
def transcribe_with_speakers(audio_file_path: str, output_csv_path: str) -> pd.DataFrame:
    """대화 음성을 화자·시각·텍스트로 정리해 DataFrame과 CSV로 반환한다."""
    with open(audio_file_path, "rb") as f:
        audio_b64 = base64.b64encode(f.read()).decode()

    audio_format = os.path.splitext(audio_file_path)[1].lstrip(".")

    response = client.chat.completions.create(
        model=AUDIO_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": INSTRUCTION},
                {"type": "input_audio", "input_audio": {"data": audio_b64, "format": audio_format}},
            ],
        }],
    )

    # 혹시 ```json ... ``` 코드블록으로 감싸서 오면 제거하고 파싱
    content = response.choices[0].message.content.strip()
    content = re.sub(r"^```(json)?|```$", "", content, flags=re.M).strip()
    data = json.loads(content)

    df = pd.DataFrame(data, columns=["start", "end", "speaker", "text"])
    df.to_csv(output_csv_path, index=False)
    return df

## 실행 / 테스트

In [4]:
audio_path = os.path.join(BASE_DIR, "data", "싼기타_비싼기타.mp3")
output_path = os.path.join(BASE_DIR, "data", "싼기타_비싼기타_result.csv")

In [5]:
df = transcribe_with_speakers(audio_path, output_path)
print(f"총 발언 수: {len(df)}\n")
print(df.to_string(index=False, max_colwidth=50))

총 발언 수: 80

 start   end speaker                                               text
   1.7  20.3     화자A 어 지금부터 저랑 그 역할극을 합시다. 역할극을 스탠딩 코미디 스타일로 할 건데 어 ...
  21.1  22.3     화자A                               어 둘 중에 어떤 역할을 맡으실래요?
  23.3  28.3     화자B           좋습니다. 그럼 제가 싼 기타로 시작하는 게 좋다는 입장을 맡아 볼게요.
  28.5  31.9     화자B                   그럼 성형님은 비싼 기타로 시작하는 게 좋다는 입장이시죠?
  32.1  33.5     화자B                                            준비되셨나요?
  33.6  34.6     화자A                                     네, 됐어요. 시작하시죠.
  35.7  36.3     화자B                                               좋아요.
  37.1  40.5     화자B                   먼저 싼 기타로 시작하는 게 좋은 이유를 말씀드리겠습니다.
  41.2  49.8     화자B 초보자일 때는 실 수도 많고 기타에 익숙해지는 과정이 필요하니까 비싼 기타보다는 부담...
  50.1  56.4     화자B        무엇보다 실력이 향상되면 그때 좋은 기타로 업그레이드 하는 것도 나쁘지 않죠?
  56.9  58.1     화자B                                    성형님은 어떻게 생각하세요?
  58.4  61.9     화자A 아, 저는 지금 말에 오페가 있다고 생각해요. 왜냐하면은 어차피 지금 비싼 기타로 나...
  62.9  69.3     화자A 그럼 어차피 비싼 기터를 사고 싼 기터도 사게 되는데 그